In [ ]:
"""
المرحلة 5.3 + 5.4 + 5.5 + 5.6:
- تدريب XGBoost مع Stratified CV (FR7, NFR1)
- SHAP Explainability (FR8)
- محرك التوصيات
- حفظ النموذج للواجهة
"""
import pandas as pd
import numpy as np
import joblib
import shap
import matplotlib.pyplot as plt
import seaborn as sns
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix

# ==========================================
# 5.3: تدريب XGBoost مع Stratified CV
# ==========================================
def train_and_evaluate(dataset_path, model_save_path="gaitly_xgboost.joblib"):
    df = pd.read_csv(dataset_path)
    
    X = df[["Mean_KneeAngle_L", "Mean_KneeAngle_R", "Cadence", "Symmetry_Knee", "Age"]]
    y = (df["Label"] == "Atypical").astype(int)  # 1 = Atypical, 0 = Typical
    
    # Stratified K-Fold (حسب NFR1: تقييم صارم)
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    
    model = XGBClassifier(
        n_estimators=100,
        max_depth=4,
        learning_rate=0.1,
        use_label_encoder=False,
        eval_metric='logloss',
        random_state=42
    )
    
    # Cross-validated predictions
    y_pred_cv = cross_val_predict(model, X, y, cv=skf)
    
    # تدريب نهائي على كل البيانات
    model.fit(X, y)
    joblib.dump(model, model_save_path)
    
    # المقاييس (NFR1: Accuracy ≥ 85%, Sensitivity ≥ 90%)
    acc = accuracy_score(y, y_pred_cv)
    prec = precision_score(y, y_pred_cv)
    rec = recall_score(y, y_pred_cv)  # Sensitivity
    f1 = f1_score(y, y_pred_cv)
    
    print("=" * 60)
    print("📊 نتائج Stratified 5-Fold Cross-Validation:")
    print("=" * 60)
    print(f"  Accuracy  : {acc:.3f}  {'✅' if acc >= 0.85 else '⚠️'} (NFR1: ≥ 0.85)")
    print(f"  Precision : {prec:.3f}")
    print(f"  Recall    : {rec:.3f}  {'✅' if rec >= 0.90 else '⚠️'} (NFR1: ≥ 0.90)")
    print(f"  F1-Score  : {f1:.3f}")
    print("=" * 60)
    print("\n📋 Classification Report:")
    print(classification_report(y, y_pred_cv, target_names=["Typical", "Atypical"]))
    
    # Confusion Matrix
    cm = confusion_matrix(y, y_pred_cv)
    plt.figure(figsize=(6, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=["Typical", "Atypical"],
                yticklabels=["Typical", "Atypical"])
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.title("Confusion Matrix (5-Fold CV)")
    plt.tight_layout()
    plt.savefig("confusion_matrix.png", dpi=150)
    plt.close()
    print("💾 تم حفظ: confusion_matrix.png")
    
    return model, X, y


# ==========================================
# 5.4: SHAP Explainability (FR8)
# ==========================================
def generate_shap_explanations(model, X, output_prefix="shap"):
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X)
    
    # 1) Summary Plot (Beeswarm)
    plt.figure()
    shap.summary_plot(shap_values, X, show=False, plot_size=(10, 6))
    plt.title("SHAP Feature Importance (Beeswarm)")
    plt.tight_layout()
    plt.savefig(f"{output_prefix}_summary.png", dpi=150, bbox_inches='tight')
    plt.close()
    
    # 2) Bar Plot (Global Importance)
    plt.figure()
    shap.summary_plot(shap_values, X, plot_type="bar", show=False)
    plt.title("SHAP Global Feature Importance")
    plt.tight_layout()
    plt.savefig(f"{output_prefix}_importance.png", dpi=150, bbox_inches='tight')
    plt.close()
    
    print(f"💾 تم حفظ: {output_prefix}_summary.png, {output_prefix}_importance.png")
    
    # 3) توليد تفسير نصي لكل عينة (للتقرير)
    feature_names = X.columns.tolist()
    explanations = []
    for i in range(len(X)):
        sample_shap = shap_values[i]
        # ترتيب الميزات حسب التأثير (مطلق)
        sorted_idx = np.argsort(np.abs(sample_shap))[::-1]
        top_features = [(feature_names[j], sample_shap[j]) for j in sorted_idx[:3]]
        explanations.append({
            "Sample_Index": i,
            "Top_Feature_1": f"{top_features[0][0]} ({top_features[0][1]:+.3f})",
            "Top_Feature_2": f"{top_features[1][0]} ({top_features[1][1]:+.3f})",
            "Top_Feature_3": f"{top_features[2][0]} ({top_features[2][1]:+.3f})"
        })
    
    expl_df = pd.DataFrame(explanations)
    expl_df.to_csv(f"{output_prefix}_per_sample.csv", index=False)
    print(f"💾 تم حفظ: {output_prefix}_per_sample.csv")
    return expl_df


# ==========================================
# 5.5: محرك التوصيات الذكي (Recommendations Engine)
# ==========================================
def generate_recommendations(feature_vector: dict, prediction: str, confidence: float) -> dict:
    """
    يولّد توصيات مبنية على القواعد (Rule-Based) مرتبطة بالخصائص المكتشفة.
    هذه ليست تشخيصاً طبياً، بل توصيات فحص إضافية.
    """
    recommendations = []
    flags = []
    
    # قاعدة 1: التماثل (Robinson Index > 10%)
    if feature_vector["Symmetry_Knee"] > 10:
        flags.append("⚠️ عدم تماثل واضح بين الجانبين")
        recommendations.append(
            "يُنصح بإجراء تقييم سريري متخصص لتقييم التماثل بين الطرفين السفليين."
        )
    
    # قاعدة 2: معدل الخطوات (Cadence خارج النطاق الطبيعي للأطفال 4-10)
    if feature_vector["Cadence"] < 90 or feature_vector["Cadence"] > 120:
        flags.append("⚠️ معدل خطوات خارج النطاق العمري المتوقع")
        recommendations.append(
            "يُنصح بمراقبة سرعة المشي وتواتر الخطوات في بيئة طبيعية."
        )
    
    # قاعدة 3: فرق الزوايا بين الجانبين
    angle_diff = abs(feature_vector["Mean_KneeAngle_L"] - feature_vector["Mean_KneeAngle_R"])
    if angle_diff > 15:
        flags.append("⚠️ فرق واضح في زوايا الركبة بين الجانبين")
        recommendations.append(
            "يُنصح بتقييم مدى حركة المفاصل (Range of Motion) من قبل أخصائي علاج طبيعي."
        )
    
    # قاعدة 4: جودة الفيديو (إذا كانت التوصية عامة)
    if not flags:
        recommendations.append(
            "المؤشرات الحركية ضمن النطاق الطبيعي المتوقع للعمر. يُنصح بمتابعة النمو الحركي بشكل دوري."
        )
    
    return {
        "Prediction": prediction,
        "Confidence": f"{confidence:.2%}",
        "Flags": flags if flags else ["✅ لا توجد مؤشرات مقلقة"],
        "Recommendations": recommendations,
        "Disclaimer": "هذه التوصيات لأغراض الفحص الأولي فقط ولا تُغني عن التقييم السريري المتخصص."
    }


# ==========================================
# 5.6: اختبار النظام على الفيديو الحقيقي
# ==========================================
def test_on_real_video(model, video_features_path="real_video_features.csv"):
    if not pd.io.common.file_exists(video_features_path):
        print("⚠️ لا يوجد فيديو حقيقي للاختبار.")
        return
    
    df = pd.read_csv(video_features_path)
    X_real = df[["Mean_KneeAngle_L", "Mean_KneeAngle_R", "Cadence", "Symmetry_Knee", "Age"]]
    
    pred = model.predict(X_real)[0]
    proba = model.predict_proba(X_real)[0]
    label = "Atypical" if pred == 1 else "Typical"
    confidence = proba[pred]
    
    feature_dict = X_real.iloc[0].to_dict()
    recs = generate_recommendations(feature_dict, label, confidence)
    
    print("\n" + "=" * 60)
    print("🎬 نتيجة تحليل الفيديو الحقيقي:")
    print("=" * 60)
    print(f"  التصنيف     : {recs['Prediction']}")
    print(f"  الثقة       : {recs['Confidence']}")
    print(f"  المؤشرات    : {recs['Flags']}")
    print(f"  التوصيات    :")
    for r in recs['Recommendations']:
        print(f"    • {r}")
    print(f"  ⚕️ تنويه      : {recs['Disclaimer']}")
    
    joblib.dump(recs, "latest_recommendations.joblib")
    print("💾 تم حفظ التوصيات في: latest_recommendations.joblib")


# ==========================================
# التشغيل الرئيسي
# ==========================================
if __name__ == "__main__":
    print("🚀 المرحلة 5.3: تدريب XGBoost...")
    model, X, y = train_and_evaluate("poc_training_dataset.csv")
    
    print("\n🚀 المرحلة 5.4: توليد تفسيرات SHAP...")
    generate_shap_explanations(model, X)
    
    print("\n🚀 المرحلة 5.6: اختبار على الفيديو الحقيقي...")
    test_on_real_video(model)
    
    print("\n✅ تم إنجاز الأسابيع 5-6 بنجاح!")

ModuleNotFoundError: No module named 'pandas'

: 